# Explainable Loan Default Risk Prediction

This notebook trains and explains a loan default risk classifier using a synthetic applicant dataset. It mirrors the project workflow from data generation through model comparison, prediction, and SHAP-based explanation.

## 1. Import Libraries

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

## 2. Generate Synthetic Loan Dataset

In [ ]:
NUMERIC_FEATURES = [
    "age",
    "annual_income",
    "employment_length_years",
    "loan_amount",
    "loan_term_months",
    "interest_rate",
    "debt_to_income_ratio",
    "credit_score",
    "credit_history_years",
    "previous_missed_payments",
]

CATEGORICAL_FEATURES = [
    "employment_status",
    "education_level",
    "loan_purpose",
    "property_area",
]

TARGET_COLUMN = "default"


def build_dataset(n_samples=1200, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)

    age = rng.integers(21, 68, n_samples)
    annual_income = rng.lognormal(mean=10.9, sigma=0.45, size=n_samples).clip(18000, 220000)
    employment_length = rng.integers(0, 28, n_samples)
    loan_amount = rng.lognormal(mean=10.2, sigma=0.5, size=n_samples).clip(2500, 85000)
    loan_term = rng.choice([36, 48, 60, 84], size=n_samples, p=[0.38, 0.2, 0.34, 0.08])
    credit_score = rng.normal(680, 72, n_samples).clip(300, 850)
    credit_history = rng.integers(1, 31, n_samples)
    missed_payments = rng.poisson(0.35, n_samples).clip(0, 6)
    debt_to_income = (loan_amount / annual_income * 0.55 + rng.normal(0.18, 0.08, n_samples)).clip(0.02, 0.75)
    interest_rate = (
        0.07
        + (760 - credit_score) / 3000
        + debt_to_income * 0.08
        + missed_payments * 0.01
        + rng.normal(0, 0.012, n_samples)
    ).clip(0.045, 0.29)

    employment_status = rng.choice(["Full-time", "Part-time", "Self-employed", "Unemployed"], n_samples, p=[0.58, 0.16, 0.18, 0.08])
    education_level = rng.choice(["High School", "Bachelor", "Master", "Doctorate"], n_samples, p=[0.32, 0.43, 0.2, 0.05])
    loan_purpose = rng.choice(["Debt consolidation", "Home improvement", "Medical", "Education", "Business"], n_samples, p=[0.42, 0.2, 0.12, 0.14, 0.12])
    property_area = rng.choice(["Urban", "Semiurban", "Rural"], n_samples, p=[0.48, 0.32, 0.2])

    logits = (
        -2.4
        + debt_to_income * 3.2
        + (loan_amount / annual_income) * 1.7
        + (680 - credit_score) / 95
        + missed_payments * 0.55
        + (interest_rate - 0.11) * 5.0
        - employment_length * 0.025
        - credit_history * 0.018
    )
    logits += np.where(employment_status == "Unemployed", 0.8, 0)
    logits += np.where(employment_status == "Part-time", 0.25, 0)
    logits += np.where(loan_purpose == "Business", 0.22, 0)
    logits += np.where(property_area == "Rural", 0.12, 0)

    default_probability = 1 / (1 + np.exp(-logits))
    default = rng.binomial(1, default_probability)

    return pd.DataFrame({
        "age": age,
        "annual_income": annual_income.round(2),
        "employment_length_years": employment_length,
        "loan_amount": loan_amount.round(2),
        "loan_term_months": loan_term,
        "interest_rate": interest_rate.round(4),
        "debt_to_income_ratio": debt_to_income.round(3),
        "credit_score": credit_score.round().astype(int),
        "credit_history_years": credit_history,
        "previous_missed_payments": missed_payments,
        "employment_status": employment_status,
        "education_level": education_level,
        "loan_purpose": loan_purpose,
        "property_area": property_area,
        "default": default,
    })


data = build_dataset()
data.head()

In [ ]:
data[TARGET_COLUMN].value_counts(normalize=True).rename("class_share")

## 3. Build Preprocessing and Model Pipelines

In [ ]:
def make_preprocessor():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ])


def make_models():
    return {
        "Logistic Regression": Pipeline([
            ("preprocessor", make_preprocessor()),
            ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
        ]),
        "Random Forest": Pipeline([
            ("preprocessor", make_preprocessor()),
            ("classifier", RandomForestClassifier(n_estimators=220, max_depth=10, class_weight="balanced", random_state=RANDOM_STATE)),
        ]),
        "Gradient Boosting": Pipeline([
            ("preprocessor", make_preprocessor()),
            ("classifier", GradientBoostingClassifier(random_state=RANDOM_STATE)),
        ]),
    }

## 4. Train and Compare Models

In [ ]:
x = data[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = data[TARGET_COLUMN]

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

models = make_models()
rows = []
confusion_matrices = {}

for name, model in models.items():
    model.fit(x_train, y_train)
    predictions = model.predict(x_test)
    probabilities = model.predict_proba(x_test)[:, 1]
    confusion_matrices[name] = confusion_matrix(y_test, predictions)
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1_score": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities),
    })

metrics = pd.DataFrame(rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
best_model_name = metrics.loc[0, "model"]
best_model = models[best_model_name]

metrics

In [ ]:
best_model_name

## 5. Visualize the Best Model Confusion Matrix

In [ ]:
matrix = confusion_matrices[best_model_name]

fig, ax = plt.subplots(figsize=(4, 3))
ax.imshow(matrix, cmap="Blues")
ax.set_xticks([0, 1], labels=["No Default", "Default"])
ax.set_yticks([0, 1], labels=["No Default", "Default"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix: {best_model_name}")

for row in range(2):
    for col in range(2):
        ax.text(col, row, int(matrix[row, col]), ha="center", va="center", color="black")

plt.tight_layout()
plt.show()

## 6. Predict Risk for a New Applicant

In [ ]:
sample_applicant = {
    "age": 36,
    "annual_income": 62000,
    "employment_length_years": 6,
    "loan_amount": 18000,
    "loan_term_months": 60,
    "interest_rate": 0.13,
    "debt_to_income_ratio": 0.28,
    "credit_score": 690,
    "credit_history_years": 9,
    "previous_missed_payments": 0,
    "employment_status": "Full-time",
    "education_level": "Bachelor",
    "loan_purpose": "Debt consolidation",
    "property_area": "Urban",
}

def risk_tier(probability):
    if probability >= 0.65:
        return "High"
    if probability >= 0.35:
        return "Medium"
    return "Low"

sample_frame = pd.DataFrame([sample_applicant])
default_probability = best_model.predict_proba(sample_frame)[0, 1]

pd.DataFrame({
    "default_probability": [default_probability],
    "risk_tier": [risk_tier(default_probability)],
})

## 7. Explain Prediction with SHAP

Run this cell after installing `shap`. The notebook keeps SHAP optional so the core model workflow can run in lighter environments.

In [ ]:
try:
    import shap

    preprocessor = best_model.named_steps["preprocessor"]
    classifier = best_model.named_steps["classifier"]
    feature_names = list(preprocessor.get_feature_names_out())

    transformed_train = preprocessor.transform(x_train)
    transformed_sample = preprocessor.transform(sample_frame)

    if hasattr(transformed_train, "toarray"):
        transformed_train = transformed_train.toarray()
    if hasattr(transformed_sample, "toarray"):
        transformed_sample = transformed_sample.toarray()

    if best_model_name in {"Random Forest", "Gradient Boosting"}:
        explainer = shap.TreeExplainer(classifier)
        shap_values = explainer.shap_values(transformed_sample)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
        elif getattr(shap_values, "ndim", 0) == 3:
            shap_values = shap_values[:, :, 1]
    else:
        explainer = shap.Explainer(classifier, transformed_train[:100])
        shap_values = explainer(transformed_sample).values

    contributions = pd.DataFrame({
        "feature": feature_names,
        "contribution": shap_values[0],
    })
    contributions["absolute_contribution"] = contributions["contribution"].abs()
    top_contributions = contributions.sort_values("absolute_contribution", ascending=False).head(12)
    display(top_contributions[["feature", "contribution"]])

    ax = top_contributions.sort_values("contribution").plot.barh(x="feature", y="contribution", figsize=(8, 5), legend=False)
    ax.set_title("Top Local SHAP Contributions")
    ax.set_xlabel("Contribution to Default Risk")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install shap to run this explanation cell: pip install shap")

## Notes

This notebook is for education and experimentation only. It should not be used for real credit decisions without validated real-world data, fairness checks, compliance review, and ongoing monitoring.